This comprehensive document is formatted as a Jupyter Notebook outline, detailing relevant Python code, implementation strategies, and theoretical background for the Machine Learning Project on Insurance Risk Analysis.

The project involves defining "Claims Risk," performing thorough **Exploratory Data Analysis (EDA)**, implementing three models (M1: Decision Tree Regressor, M2: Feed-Forward Neural Network Regressor, M3: Method of Choice, e.g., XGBoost), with M1 and M2 requiring implementations *from scratch* (using only NumPy/SciPy) and *reference implementations* (using libraries).

***

# Machine Learning Project: Insurance Risk Analysis

## 1. Project Setup and Data Loading

This section covers the necessary Python library imports and initial steps for loading and inspecting the insurance risk data.

### 1.1 Library Imports (Code Cell)

The project will rely heavily on standard numerical and machine learning libraries, including `pandas` for data manipulation, `numpy` for mathematical operations, `matplotlib` for visualization, and `scikit-learn` (sklearn) for preprocessing, modeling (reference models), and evaluation. For M2 (FFNN reference model), `TensorFlow` and `Keras` are required.

```python
import numpy as np # Standard numerical library
import pandas as pd # Data manipulation library
import matplotlib.pyplot as plt # Standard visualization library

# Scikit-learn essential modules
from sklearn.model_selection import train_test_split # For splitting data
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Feature scaling and encoding
from sklearn.impute import SimpleImputer # Handling missing values
from sklearn.pipeline import Pipeline, make_pipeline, ColumnTransformer # Building transformation workflows
from sklearn.metrics import mean_squared_error, mean_absolute_error # Evaluation metrics (RMSE, MAE)

# EDA and Unsupervised Learning tools
from sklearn.decomposition import PCA # Principal Component Analysis
from sklearn.cluster import KMeans # Clustering
from sklearn.ensemble import RandomForestRegressor # Useful ensemble model

# Model Implementations (Reference/M3)
from sklearn.tree import DecisionTreeRegressor # M1 reference
from sklearn.neural_network import MLPRegressor # M2 Scikit-learn reference
# Note: XGBoost (M3 choice) requires 'import xgboost as xgb' (library not detailed in snippets, but referenced)

# Keras/TensorFlow for M2 Deep Learning reference
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.metrics import RootMeanSquaredError

# Tool for saving models
import joblib # For saving Scikit-learn models/pipelines
```

### 1.2 Loading Data (Conceptual Implementation)

The source mentions the dataset contains $\sim 68000$ entries with 12 columns, detailing vehicular insurance policies and claims history. Assuming the data is provided as a CSV file (e.g., `insurance_claims.csv`).

```python
# Assuming data loading from a CSV file
# data = pd.read_csv('insurance_claims.csv')
# print(data.head())
# data.info() # Initial check of data types and non-null counts
```

## 2. Data Preparation: Definition and Splitting

### 2.1 Defining the Target Variable (Theory/Implementation)

The project requires defining a reasonable measure for "claims risk" to be treated as the target variable. The report mentions defining the target variable as **"Claims Risk"**. Since the required models are regressors (M1, M2), the target variable will likely be quantitative, such as the total claim amount or frequency/severity measures.

**Theory:** Regression tasks involve predicting values. The target variable needs to be clearly defined and argued for in the report.

**Implementation snippet (Conceptual):**

```python
# Example: Defining the target variable and separating predictors/labels
# target_variable_name = "claims_risk_measure" # E.g., Claims amount, or calculated ratio
# X = data.drop(columns=[target_variable_name])
# y = data[target_variable_name].copy()

# Create a test set and set it aside immediately to avoid data snooping bias
# X_train_full, X_test, y_train_full, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42)
```

## 3. Data Cleaning and Exploratory Data Analysis (EDA)

EDA is crucial for illustrating selected aspects of the data and arguing which features require cleaning.

### 3.1 Initial Data Inspection and Cleaning (Code/Theory)

**Theory:** Initial inspection identifies feature types, missing values, and data distribution. Missing features must be handled, typically by removal or imputation (e.g., zero, mean, or median).

```python
# Check for missing values in the full dataset (before splitting, or on the training set)
# print(data.isnull().sum()) 

# Imputation using SimpleImputer (e.g., replacing numerical missing values with the median)
imputer_median = SimpleImputer(strategy="median")

# If dealing with categorical features missing values, use a strategy like 'most_frequent'
imputer_mode = SimpleImputer(strategy="most_frequent")

# We can define a function transformer for feature engineering, such as creating ratios
def column_ratio(X):
    # X[:, ] / X[:,] divides the first column by the second column
    return X[:, ] / X[:,]

# Example of dropping rows with missing values (if appropriate for the scale of missingness)
# data_cleaned = data.dropna()
```

### 3.2 Feature Engineering and Scaling (Code/Theory)

**Theory:** Feature engineering creates new features or transforms existing ones (e.g., computing ratios like rooms per household) to improve correlation with the target variable. **Feature Scaling (Standardization/Normalization)** is critical, especially for models based on gradient descent, like M2 (FFNN).

```python
# Example Feature Engineering (based on California housing context, adapted conceptually)
# Assuming 'Rooms', 'Households', and 'Population' are present:
# data["rooms_per_house"] = data["Rooms"] / data["Households"]
# data["people_per_house"] = data["Population"] / data["Households"]

# Feature Scaling: Standardizing numerical features (recommended for models sensitive to scale)
scaler = StandardScaler()

# Handling Categorical Features: One-Hot Encoding (for ML algorithms that only accept numerical inputs)
one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False) 
```

### 3.3 Visualization (Code/Theory)

#### 3.3.1 Visualization of Correlations/Distributions (Code/Theory)

**Theory:** Visualizing distributions (histograms) helps identify skewed data requiring transformation (e.g., logarithmic transformation). Correlation analysis helps understand relationships between attributes and the target.

```python
# Visualize distributions (Histograms)
# data.hist(bins=50, figsize=(12, 8))
# plt.show()

# Look for correlations with the target variable
# correlation_matrix = data.corr()
# print(correlation_matrix[target_variable_name].sort_values(ascending=False)) 
```

#### 3.3.2 Principal Component Analysis (PCA) for Visualization (Code/Theory)

**Theory:** PCA is a dimensionality reduction technique used to visualize the overall structure of the data by projecting it onto a lower-dimensional space (e.g., 2D). This helps reveal hidden patterns or structure in the dataset.

**Implementation Steps:** PCA requires numerical, typically scaled, input data.

```python
# 1. Scale the numerical data first
# X_scaled = scaler.fit_transform(X_numerical) 

# 2. Apply PCA to reduce dimensions to 2
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_scaled) 

# 3. Plot the PCA results
# plt.figure(figsize=(8, 6))
# plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=y, cmap='viridis') # Color based on target variable
# plt.xlabel("Principal Component 1")
# plt.ylabel("Principal Component 2")
# plt.title("PCA of Policy Features")
# plt.show()
```

#### 3.3.3 Clustering (K-Means) for Visualization/Segmentation (Code/Theory)

**Theory:** Clustering algorithms like **K-Means** identify natural segments (subgroups) of policyholders based on their features. The cluster assignments can be used to color the PCA plot for further insight.

```python
# 1. Choose k (number of clusters), possibly using methods like silhouette coefficient
k = 4 # Example value
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

# 2. Fit K-Means on the scaled data and predict cluster assignments
# cluster_labels = kmeans.fit_predict(X_scaled)

# 3. Use clusters to color the PCA plot for insight
# plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=cluster_labels, cmap='jet')
# plt.title("PCA colored by K-Means Clusters")
# plt.show()
```

## 4. Model Implementation M1: Decision Tree Regressor

### 4.1 Theory: Decision Tree Regressor

A Decision Tree Regressor partitions the data into branches to create nodes with **minimal variance**. The underlying algorithm is often CART (Classification and Regression Trees). The objective at each split is to find the feature and threshold that results in the **greatest reduction in Mean Squared Error (MSE)**. Decision trees are known for being highly interpretable ("white box" models).

### 4.2 M1 Implementation (From Scratch - Conceptual)

The scratch implementation must use only standard libraries, NumPy, and SciPy. While full implementation is complex, the core logic involves iteratively calculating splits based on maximizing MSE reduction. This primarily requires NumPy operations.

```python
# Python/NumPy Code Snippet (Conceptual - Focus on core calculation/data structure)
# This snippet illustrates core mathematical operations needed in scratch implementation,
# focusing only on NumPy/array manipulation, NOT using sklearn tree logic.

# Assume 'X' is a NumPy array of features and 'y' is the target array.
# Function to calculate variance/MSE for a subset of the target values 'y_subset'
def calculate_mse(y_subset):
    if len(y_subset) == 0:
        return 0
    # For regression, the predicted value is the mean of y_subset
    mean = np.mean(y_subset)
    # MSE is the variance of the residuals (y_i - mean)^2
    return np.mean((y_subset - mean) ** 2)

# Function to evaluate a split (e.g., using a threshold 't' on feature 'j')
def evaluate_split(X, y, j, t):
    left_indices = X[:, j] <= t
    right_indices = X[:, j] > t
    
    y_left, y_right = y[left_indices], y[right_indices]
    
    # Calculate MSE reduction (Impurity Reduction)
    mse_total = calculate_mse(y)
    mse_left = calculate_mse(y_left)
    mse_right = calculate_mse(y_right)
    
    # Calculate weighted MSE after split
    n_total = len(y)
    n_left = len(y_left)
    n_right = len(y_right)
    
    weighted_mse = (n_left / n_total) * mse_left + (n_right / n_total) * mse_right
    
    # Reduction in MSE is mse_total - weighted_mse (Maximize this reduction)
    return mse_total - weighted_mse

# Core CART algorithm needs loops to iterate through features (j) and thresholds (t)
# (Implementation requires building tree nodes recursively)
# ... Implementation of Node Class, Splitting logic, and Tree Builder ...
```

### 4.3 M1 Reference Implementation (Code/Implementation)

The reference implementation uses `sklearn.tree.DecisionTreeRegressor`.

```python
# Use the reference implementation for M1
dt_reg = DecisionTreeRegressor(max_depth=5, random_state=42)

# Train the model
# dt_reg.fit(X_train_processed, y_train)

# Making predictions
# y_pred_dt = dt_reg.predict(X_test_processed)
# rmse_dt = mean_squared_error(y_test, y_pred_dt, squared=False)
```

**Implementation Note:** The project requires handling at least one **categorical variable** in M1. This means ensuring that categorical features are encoded numerically (e.g., using OneHotEncoder) *before* feeding data into the tree model, or using an algorithm designed for categorical inputs (like Histogram-Based Gradient Boosting, which handles them natively, although M1 specifies Decision Tree Regressor).

## 5. Model Implementation M2: Feed-Forward Neural Network Regressor

### 5.1 Theory: Feed-Forward Neural Network (FFNN) Regressor

An FFNN is organized in layers of neurons, where information flows in one direction (feed-forward). For regression, the network aims to minimize a loss function (like MSE) using the **backpropagation algorithm** to iteratively update weights and biases. The output layer typically has a single neuron with a linear activation function.

### 5.2 M2 Implementation (From Scratch - Conceptual)

The scratch implementation must use only standard libraries, NumPy, and SciPy. This involves implementing forward propagation, calculating the loss, implementing backpropagation (calculating gradients), and updating weights via an optimizer (e.g., Stochastic Gradient Descent - SGD).

```python
# Python/NumPy/SciPy Code Snippet (Conceptual - Focus on optimization using SGD/NumPy)
# Based on SGD algorithm structure for parameter optimization

# Assume 'theta' represents the weights and biases
# Assume 'X_b' includes the bias feature (x0=1)
# Assume 'm' is the number of instances in the training set

# Define learning schedule (critical for SGD convergence)
# def learning_schedule(t):
#     return t0 / (t + t1) 

# For loop representing epochs and iterations (batches)
# for epoch in range(n_epochs):
#     for iteration in range(m):
#         X_batch, y_batch = select_random_batch() # Batch selection logic
#         
#         # Forward pass (requires implementing matrix multiplications and activation functions manually)
#         # y_pred = FFNN_forward(X_batch, theta)
#         
#         # Compute gradients (Backpropagation)
#         # gradients = calculate_gradients(X_batch, y_batch, y_pred, theta) 
#         
#         # Update learning rate
#         # eta = learning_schedule(epoch * m + iteration)
#         
#         # Update weights (theta)
#         # theta = theta - eta * gradients
```

### 5.3 M2 Reference Implementation (Code/Implementation)

The reference implementation uses Keras/TensorFlow, which is the preferred high-level API for deep learning.

```python
# 1. Define the model using Keras Sequential API
tf.random.set_seed(42)

model = Sequential([
    Input(shape=(X_train_processed.shape,)), # Input layer definition
    Dense(100, activation="relu", kernel_initializer="he_normal"), # Hidden layer 1 (ReLU activation, He initialization recommended)
    Dense(50, activation="relu", kernel_initializer="he_normal"),  # Hidden layer 2
    Dense(1, activation="linear") # Output layer for regression (single neuron, linear activation)
])

# 2. Compile the model
# We use MSE loss (common for regression) and RootMeanSquaredError metric
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", 
              optimizer=optimizer, 
              metrics=[RootMeanSquaredError()])

# 3. Train the model
# history = model.fit(X_train_processed, y_train, epochs=20, 
#                     validation_data=(X_valid_processed, y_valid))

# 4. Alternative Scikit-learn reference (less flexible for deep learning, but simple to use)
# mlp_reg = MLPRegressor(hidden_layer_sizes=, activation='relu', solver='adam', random_state=42)
```

## 6. Model Implementation M3: Method of Choice (XGBoost)

### 6.1 Theory: XGBoost (e.g., Gradient Boosting)

Gradient Boosting is an ensemble technique that builds models sequentially, where each new model corrects the errors (residuals) of the previous ones. **XGBoost** (Extreme Gradient Boosting) is a highly optimized and scalable implementation of gradient boosting that often yields state-of-the-art results on tabular data.

### 6.2 M3 Implementation (Code/Implementation)

XGBoost is implemented using its dedicated library (not part of Scikit-learn/TensorFlow core, though Scikit-learn offers `GradientBoostingRegressor` and `HistGradientBoostingRegressor`). Assuming XGBoost is the choice for M3:

```python
# Import XGBoost (conceptual placeholder, requires separate installation)
# import xgboost as xgb 
# from xgboost import XGBRegressor 

# Alternative Scikit-learn Gradient Boosting Regressor (Reference implementation)
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting Implementation (Library Based)
gbrt = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
# gbrt.fit(X_train_processed, y_train)

# If using XGBoost:
# xgb_reg = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
# xgb_reg.fit(X_train_processed, y_train)
```

## 7. Model Evaluation and Persistence

### 7.1 Evaluation Metrics (Theory/Implementation)

The project requires reporting and arguing for evaluation metrics. Common metrics for regression are:

1.  **Root Mean Squared Error (RMSE):** Measures the standard deviation of the residuals (prediction errors). It is heavily influenced by large errors (outliers).
2.  **Mean Absolute Error (MAE):** Measures the average magnitude of the errors. Less sensitive to outliers than RMSE.

```python
# Calculating RMSE (using Scikit-learn)
# rmse = mean_squared_error(y_test, y_pred, squared=False) 

# Calculating MAE (using Scikit-learn)
# mae = mean_absolute_error(y_test, y_pred) 

# Example of comparing performance (Discussion required in report)
# comparison_table = pd.DataFrame({
#     'Model': ['DT', 'FFNN', 'XGBoost'],
#     'RMSE': [rmse_dt, rmse_ffnn, rmse_xgb],
#     'MAE': [mae_dt, mae_ffnn, mae_xgb]
# })
# print(comparison_table)
```

### 7.2 Model Persistence (Code/Implementation)

Once the best models are fine-tuned, they should be saved for later use or deployment (launching).

**Theory:** `joblib` is commonly used for saving Scikit-learn models/pipelines. Keras models use the TensorFlow SavedModel format.

```python
# Saving Scikit-learn Models (M1/M3/Preprocessing Pipelines) using joblib
# joblib.dump(final_pipeline, "claims_risk_pipeline.pkl")
# joblib.dump(best_dt_model, "best_dt_model.pkl")

# Loading Scikit-learn Models
# reloaded_pipeline = joblib.load("claims_risk_pipeline.pkl")
# reloaded_model = joblib.load("best_dt_model.pkl")

# Saving Keras/TensorFlow Models (M2)
# best_ffnn_model.save("best_ffnn_model", save_format="tf")

# Loading Keras/TensorFlow Models
# reloaded_ffnn_model = tf.keras.models.load_model("best_ffnn_model")
```

## 8. Hyperparameter Tuning Snippets

Hyperparameter selection and discussion are mandatory parts of the report. Grid Search and Randomized Search are common tuning techniques.

```python
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Example of parameters for Decision Tree (M1)
# param_grid_dt = {
#     'max_depth':,
#     'min_samples_leaf':
# }

# grid_search_dt = GridSearchCV(DecisionTreeRegressor(random_state=42), 
#                              param_grid_dt, 
#                              cv=5, 
#                              scoring='neg_mean_squared_error')
# grid_search_dt.fit(X_train_processed, y_train)
# best_dt_params = grid_search_dt.best_params_

# Note on Keras Tuning: The Keras Tuner library is recommended for advanced DNN tuning, 
# which involves defining a kt.HyperModel class or using GridSearchCV wrapper 
# if the Scikit-learn API compatibility is required.
```

## 9. Comprehensive Data Transformation Pipeline

To ensure reproducibility and ease of testing various transformations, using a Scikit-learn `Pipeline` and `ColumnTransformer` is highly recommended. This encompasses cleaning, engineering, and scaling (as noted in implementation guidance).

**Theory:** A `ColumnTransformer` handles different preprocessing steps for different columns (e.g., numerical vs. categorical features). A `Pipeline` chains transformers and a final estimator.

```python
# Assume X contains numerical features (e.g., Exposure, VehicleAge) and 
# categorical features (e.g., VehicleMake, Region)

numerical_features = ['Exposure', 'VehicleAge', 'OtherNumFeature']
categorical_features = ['VehicleMake', 'Region', 'OtherCatFeature']
ratio_features = ['Exposure', 'Population'] # Example features for ratio creation

# Pipeline for Numerical Features (Imputation + Scaling)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler())
])

# Pipeline for Categorical Features (Imputation + One-Hot Encoding)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Pipeline for creating ratio feature (Example, requires adjustment based on dataset structure)
# ratio_pipeline = make_pipeline(
#     SimpleImputer(strategy="median"),
#     FunctionTransformer(column_ratio, feature_names_out='one-to-one'), 
#     StandardScaler()
# )


# Combine transformations using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_features),
        ('cat', cat_pipeline, categorical_features),
        # Add ratio features pipeline here if needed
    ],
    remainder='passthrough' # Keep any remaining features untouched
)

# Full Model Pipeline Example (for M3, using Gradient Boosting)
# final_pipeline_gbr = Pipeline([
#     ('preprocessing', preprocessor),
#     ('regressor', GradientBoostingRegressor(random_state=42))
# ])

# Note: All training data (X_train_full) should be fit_transformed through the pipeline 
# before training library-based models (M1 reference, M2 reference, M3), 
# or manually preprocessed for scratch implementations.
```

### LINEAR REGRSSION 

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.feature_selection import f_regression
from sklearn.metrics import mean_squared_error, r2_score

X = "dataset"
y = "Target"
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Print coefficients
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

### Linear regression Feature Selection

In [ ]:
# Function to evaluate model and return F-statistic and p-value
def evaluate_model(X, y):
    f_stats, p_values = f_regression(X, y)
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return f_stats, p_values, mse, r2

# Backward Elimination
features_backward = "list of features"
while True:
    f_stats, p_values, _, _ = evaluate_model(df[features_backward], y)
    max_p = max(p_values)
    if max_p > 0.05:
        idx = p_values.tolist().index(max_p)
        del features_backward[idx]
    else:
        break
f_stats_backward, p_values_backward, mse_backward, r2_backward = evaluate_model(df[features_backward], y)

# Forward Selection
features_forward = []
remaining_features = feature_names.copy()
while remaining_features:
    best_feature = None
    best_p = 1
    for feature in remaining_features:
        test_features = features_forward + [feature]
        f_stats, p_values, _, _ = evaluate_model(df[test_features], y)
        if p_values[-1] < best_p:
            best_p = p_values[-1]
            best_feature = feature
    if best_p < 0.05:
        features_forward.append(best_feature)
        remaining_features.remove(best_feature)
    else:
        break
f_stats_forward, p_values_forward, mse_forward, r2_forward = evaluate_model(df[features_forward], y)

# Stepwise Selection
features_stepwise = []
remaining_features = feature_names.copy()
while True:
    # Forward step
    best_feature = None
    best_p = 1
    for feature in remaining_features:
        test_features = features_stepwise + [feature]
        f_stats, p_values, _, _ = evaluate_model(df[test_features], y)
        if p_values[-1] < best_p:
            best_p = p_values[-1]
            best_feature = feature
    if best_p < 0.05:
        features_stepwise.append(best_feature)
        remaining_features.remove(best_feature)
    else:
        break
    # Backward step
    f_stats, p_values, _, _ = evaluate_model(df[features_stepwise], y)
    for i in range(len(features_stepwise)):
        if p_values[i] > 0.05:
            del features_stepwise[i]
            break
f_stats_stepwise, p_values_stepwise, mse_stepwise, r2_stepwise = evaluate_model(df[features_stepwise], y)

# Print results

### Cross Validation, Ridge Lasso

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import KFold

from sklearn.preprocessing import StandardScaler
#split Data
def split_data(n, n_samples, seed_number):
    random.seed(seed_number)
    train_idx, val_idx = randomize_split(n, n_samples)

    train_df = data.iloc[train_idx, :]
    val_df = data.iloc[val_idx, :]
    return train_df, val_df


n = len(data)
n_samples = int(n/2)

train_df, val_df = split_data(n, n_samples, 30)
print(len(train_df), len(val_df))


#Leave one out 
def split_data(n, n_samples, seed_number):
    random.seed(seed_number)
    train_idx, val_idx = randomize_split(n, n_samples)

    train_df = data.iloc[train_idx, :]
    val_df = data.iloc[val_idx, :]
    return train_df, val_df


n = len(data)
n_samples = int(n/2)

train_df, val_df = split_data(n, n_samples, 30)
print(len(train_df), len(val_df))

#K-Fold codes. 
def kfold_mse(df, x_col, y_col, k):
    X = df[[x_col]].values if isinstance(x_col, str) else df[x_col].values
    y = df[y_col].values
    n = len(df)
    
    # Shuffle the indices
    indices = np.arange(n)
    #np.random.seed(42)
    np.random.shuffle(indices)
    
    fold_sizes = [n // k] * k
    for i in range(n % k):
        fold_sizes[i] += 1  # distribute remainder
    
    current = 0
    y_pred = np.zeros_like(y, dtype=float)
    
    for fold_size in fold_sizes:
        val_idx = indices[current:current + fold_size]
        train_idx = np.setdiff1d(indices, val_idx)
        
        X_train, X_val = X[train_idx], X[val_idx]
        y_train = y[train_idx]
        
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred[val_idx] = model.predict(X_val)  # assign whole array
        
        current += fold_size
    
    mse = mean_squared_error(y, y_pred)
    return mse

for k in [5, 8, 10]:
    mse = kfold_mse(data, x_col='Heart_Rate', y_col='Calories', k=k)
    print(f"{k}-fold CV MSE: {mse:.4f}")



## PCA 

In [ ]:

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import pandas as pd


# Perform PCA to reduce to 3 components
X_reduced = PCA(n_components=3).fit_transform("Dataset")

# Create a DataFrame with PCA components and target labels
df_pca = pd.DataFrame(X_reduced, columns=["PC1", "PC2", "PC3"])
df_pca["target"] = "targt"


# Create 2D scatter plots for each pair of principal components
plt.figure(figsize=(18, 5))

# PC1 vs PC2
plt.subplot(1, 3, 1)
sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue=)
plt.title("PC1 vs PC2")

# PC1 vs PC3
plt.subplot(1, 3, 2)
sns.scatterplot(data=df_pca, x="PC1", y="PC3", hue=)
plt.title("PC1 vs PC3")

# PC2 vs PC3
plt.subplot(1, 3, 3)
sns.scatterplot(data=df_pca, x="PC2", y="PC3", hue="")
plt.title("PC2 vs PC3")

plt.tight_layout()
plt.show()


pca = PCA(n_components=3)
pca.fit("data")


fig = plt.figure(1, figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d", elev=-150, azim=110)

X_reduced = PCA(n_components=3).fit_transform("data" ) #data wihout the target. 
scatter = ax.scatter(
    X_reduced[:, 0],
    X_reduced[:, 1],
    X_reduced[:, 2],
    c="data",
    s=40,
)

ax.set(
    title="First three PCA dimensions",
    xlabel="1st Eigenvector",
    ylabel="2nd Eigenvector",
    zlabel="3rd Eigenvector",
)
ax.xaxis.set_ticklabels([])
ax.yaxis.set_ticklabels([])
ax.zaxis.set_ticklabels([])

# Add a legend
legend1 = ax.legend(
    scatter.legend_elements()[0],
    loc="upper right",
    title="Classes",
)
ax.add_artist(legend1)

plt.show()


## DECISION TREE BUILT FROM SCRATCH

In [ ]:
X_train_scratch, X_val_scratch, y_train_scratch, y_val_scratch = train_test_split(X, y, test_size=0.2, random_state=42)
class Node:
    """
    A node in the decision tree.
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature        # Index of feature to split on
        self.threshold = threshold    # Threshold value for the split
        self.left = left              # Left child node
        self.right = right            # Right child node
        self.value = value            # Class label for leaf nodes
    
    def is_leaf(self):
        """Check if the node is a leaf node"""
        return self.value is not None


# Define the Decision Tree Classifier class
class DecisionTreeFromScratch:
    """
    A Decision Tree classifier built from scratch.
    """
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    def fit(self, X, y):
        """Build the decision tree"""
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _build_tree(self, X, y, depth):
        """
        Recursively build the decision tree.
        """
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_classes == 1 or \
           n_samples < self.min_samples_split:
            # Create a leaf node with the most common class
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        # Find the best split
        best_feature, best_threshold = self._best_split(X, y)
        
        # If no split improves the tree, create a leaf node
        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        # Split the dataset
        left_indices = X[:, best_feature] <= best_threshold
        right_indices = X[:, best_feature] > best_threshold
        
        # Recursively build left and right subtrees
        left_child = self._build_tree(X[left_indices], y[left_indices], depth + 1)
        right_child = self._build_tree(X[right_indices], y[right_indices], depth + 1)
        
        return Node(feature=best_feature, threshold=best_threshold, 
                   left=left_child, right=right_child)
    
    def _best_split(self, X, y):
        """
        Find the best split for the dataset.
        """
        n_samples, n_features = X.shape
        
        if n_samples <= 1:
            return None, None
        
        best_gini = float('inf')
        best_feature = None
        best_threshold = None
        
        # Try all features
        for feature_idx in range(n_features):
            X_col = X[:, feature_idx]
            thresholds = np.unique(X_col)
            
            # Try all possible thresholds
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i + 1]) / 2
                
                # Split the data
                left_indices = X_col <= threshold
                right_indices = X_col > threshold
                
                left_y = y[left_indices]
                right_y = y[right_indices]
                
                # Calculate weighted Gini impurity
                w_gini = weighted_impurity(left_y, right_y, gini_impurity)
                
                # Update best split if this is better
                if w_gini < best_gini:
                    best_gini = w_gini
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _most_common_label(self, y):
        """Return the most common label in y"""
        unique, counts = np.unique(y, return_counts=True)
        return unique[np.argmax(counts)]
    
    def predict(self, X):
        """Predict class labels for samples in X"""
        return np.array([self._traverse_tree(x, self.root) for x in X])
    
    def _traverse_tree(self, x, node):
        """
        Traverse the tree to make a prediction for a single sample.
        """
        # If we're at a leaf node, return its value
        if node.is_leaf():
            return node.value
        
        # Otherwise, traverse left or right based on the feature value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

dt_scratch = DecisionTreeFromScratch(max_depth=10, min_samples_split=2)
dt_scratch.fit(X_train_scratch, y_train_scratch)

# Make predictions
y_train_pred_scratch = dt_scratch.predict(X_train_scratch)
y_val_pred_scratch = dt_scratch.predict(X_val_scratch)

# Evaluate the classifier
train_accuracy_scratch = accuracy_score(y_train_scratch, y_train_pred_scratch)
val_accuracy_scratch = accuracy_score(y_val_scratch, y_val_pred_scratch)

print(f"Training Accuracy: {train_accuracy_scratch:.4f}")
print(f"Validation Accuracy: {val_accuracy_scratch:.4f}")
print("\nConfusion Matrix (Validation):")
print(confusion_matrix(y_val_scratch, y_val_pred_scratch))
print("\nClassification Report (Validation):")
print(classification_report(y_val_scratch, y_val_pred_scratch))


print(f"From-scratch validation accuracy: {val_accuracy_scratch:.4f}")
print(f"sklearn validation accuracy: {val_accuracy:.4f}")
print(f"Difference: {abs(val_accuracy_scratch - val_accuracy):.4f}")

# Plot decision boundaries side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot from-scratch tree
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# From-scratch predictions
Z_scratch = dt_scratch.predict(np.c_[xx.ravel(), yy.ravel()])
Z_scratch = Z_scratch.reshape(xx.shape)

axes[0].contourf(xx, yy, Z_scratch, alpha=0.4, cmap='coolwarm')
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=50)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title(f'From-Scratch Tree (Acc: {val_accuracy_scratch:.2f})')
axes[0].grid(True, alpha=0.3)

# sklearn predictions
Z_sklearn = dt_classifier.predict(np.c_[xx.ravel(), yy.ravel()])
Z_sklearn = Z_sklearn.reshape(xx.shape)

axes[1].contourf(xx, yy, Z_sklearn, alpha=0.4, cmap='coolwarm')
axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=50)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title(f'sklearn Tree (Acc: {val_accuracy:.2f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()